In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import joblib
import pandas as pd
import chess

from chesswinnerprediction.static_move.constants import STATIC_MOVE_COLUMNS
from chesswinnerprediction.static_move.processing.utils import process_df

In [5]:
static_move_demo_df = pd.read_csv(
    "/home/tikhon/PycharmProjects/ChessWinnerPrediction/data/processed/demo/static_move.csv"
)
baseline_demo_df = pd.read_csv(
    "/home/tikhon/PycharmProjects/ChessWinnerPrediction/data/processed/demo/baseline.csv"
)

In [6]:
static_move_model = joblib.load("/home/tikhon/PycharmProjects/ChessWinnerPrediction/models/static_move.pkl")
baseline_model = joblib.load("/home/tikhon/PycharmProjects/ChessWinnerPrediction/models/baseline.pkl")

In [10]:
def static_move_prediction(game):
    X_data: pd.DataFrame = process_df(game[STATIC_MOVE_COLUMNS])
    X_data.drop(columns=["Event", "Result"], inplace=True)
    predictions = static_move_model.predict_proba(X_data)
    return predictions

def get_baseline_prediction(data):
    X_data = data.drop(columns=["GameId"])
    predictions = baseline_model.predict_proba(X_data)
    return predictions

def process_game(game):
    board = chess.Board()
    game_dict = dict()
    st_predictions = static_move_prediction(game)
    for ((_, (i_move, move_san)), predict) in zip(game[["i_move", "chess_moves_list"]].iterrows(), st_predictions):
        move = board.parse_san(move_san)
        board.push(move)
        game_dict[i_move] = {
            "board": board.copy(),
            "static_move_prediction": predict,
        }
    return game_dict

def get_games(static_move_data_df, baseline_data_df):
    games = dict()
    
    game_ids = static_move_data_df["GameId"].unique()
    for game_id in game_ids:
        x_baseline = baseline_data_df[baseline_data_df["GameId"] == game_id]
        baseline_prediction = get_baseline_prediction(x_baseline)
        
        x_static_move = static_move_data_df[static_move_data_df["GameId"] == game_id]
        game_processed = process_game(x_static_move)
        
        games[game_id] = {
            "baseline_prediction": baseline_prediction,
            "game": game_processed
        }
    return games

In [11]:
games_dict = get_games(static_move_demo_df, baseline_demo_df)

In [12]:
games_dict

{2129: {'baseline_prediction': array([[0.42711933, 0.41868654, 0.15419413]]),
  'game': {1: {'board': Board('rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq - 0 1'),
    'static_move_prediction': array([0.50042722, 0.46604309, 0.03352969])},
   2: {'board': Board('rnbqkbnr/pppppp1p/6p1/8/3P4/8/PPP1PPPP/RNBQKBNR w KQkq - 0 2'),
    'static_move_prediction': array([0.46876096, 0.49921482, 0.03202422])},
   3: {'board': Board('rnbqkbnr/pppppp1p/6p1/8/2PP4/8/PP2PPPP/RNBQKBNR b KQkq - 0 2'),
    'static_move_prediction': array([0.49559744, 0.47271814, 0.03168443])},
   4: {'board': Board('rnbqk1nr/ppppppbp/6p1/8/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 1 3'),
    'static_move_prediction': array([0.47684214, 0.49080369, 0.03235417])},
   5: {'board': Board('rnbqk1nr/ppppppbp/6p1/8/2PP4/2N5/PP2PPPP/R1BQKBNR b KQkq - 2 3'),
    'static_move_prediction': array([0.50285364, 0.4639581 , 0.03318826])},
   6: {'board': Board('rnbqk2r/ppppppbp/5np1/8/2PP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 3 4'),
    'stati